# Verification of Reported Statistics

Recomputes the statistics reported in the published paper (Barumerli et al., *Acta Acustica* 2026,
10, 89, doi:10.1051/aacus/2026084) from the cached results in `results/` and compares them with the
published values. It only reads CSV files, so it runs in seconds without HRTFs or behavioural data.

**Contexts:**
1. Parameter recovery (Sec. 3.3, Fig. S3) — Pearson r and mean bias
1b. Template reconstruction error (Sec. 3.2) — RMSE per interpolation method
2. Behavioural-metric recovery (Sec. 3.3) — Pearson r (from the saved output of notebook 04)
3. Fitted parameters (Sec. 3.4, Table 1 top panel) — mean ± SE, ranges, Friedman tests, Spearman ρ
4. Posterior predictive checks (Sec. 3.4.1, Table 1 middle/bottom panels, Fig. 5) — Pearson r, Wilcoxon tests
5. Model comparison (Sec. 3.5, Table 2, Fig. 6) — ΔBIC, Friedman test, win counts

Pearson r, Spearman ρ and Friedman tests follow `R/fitting_analysis.Rmd` and notebooks 04 and 08;
Wilcoxon signed-rank tests use `scipy.stats.wilcoxon` with its defaults (exact distribution), as for
the published values.

Each value is marked ✓ (matches at the published precision), ✗ (does not match) or **note** (a
known, documented difference between the published text and the analysis, e.g. a value reported
as mean ± SE that is actually mean ± SD). The notes are listed in the README
("Notes on the published values"); none of them changes a conclusion of the paper.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

import paths

# === Load all cached results ===
recovery_df = pd.read_csv(paths.RESULTS_DIR / 'parameter_recovery_results.csv')
fitting_df = pd.read_csv(paths.FITTING_RESULTS)
comparison_df = pd.read_csv(paths.RESULTS_DIR / 'synthetic_vs_actual_comparison.csv')
actual_df = pd.read_csv(paths.RESULTS_DIR / 'actual_behavioural_metrics.csv')

print(f'Parameter recovery: {len(recovery_df)} rows (N={len(recovery_df)} combos)')
print(f'Fitting results:    {len(fitting_df)} rows ({len(fitting_df["participant"].unique())} participants x {len(fitting_df["method"].unique())} methods)')
print(f'Comparison:         {len(comparison_df)} rows')
print(f'Actual behavioral:  {len(actual_df)} participants')

methods = ['SHMAX', 'SH', 'barycentric', 'barumerli2023']

# Collector for the summary table
summary_rows = []


def check(section, quantity, computed, published, decimals=2):
    """Compare a computed value with the published one at the published precision."""
    # tolerance: half a unit of the last published digit (+2% for round-half ties, e.g. 0.750 -> 0.8)
    match = abs(float(computed) - published) <= 10 ** -decimals * 0.51
    summary_rows.append({'Section': section, 'Quantity': quantity,
                         'Published': published, 'Computed': round(float(computed), decimals + 1),
                         'Match': '✓' if match else '✗'})
    return match


def note(section, quantity, published, computed, explanation):
    """Record a documented difference between the published text and the analysis."""
    summary_rows.append({'Section': section, 'Quantity': quantity, 'Published': published,
                         'Computed': computed, 'Match': 'note', 'Note': explanation})


Parameter recovery: 28 rows (N=28 combos)
Fitting results:    132 rows (33 participants x 4 methods)
Comparison:         165 rows
Actual behavioral:  33 participants


## Context 1 — Parameter Recovery (Sec. 3.3, Fig. S3)

Source: `04_parameter_recovery.ipynb` (`results/parameter_recovery_results.csv`).

Published:
- σ_m r = 0.97 (mean bias −0.1°), σ_mon r = 0.85 (+3.0 dB), σ_prior r = 0.84 (+19.9°); all p < 0.001
- recovered parameters uncorrelated: r = −0.01, p = .95 (σ_m vs σ_mon); r = −0.16, p = .42 (σ_mon vs σ_prior)

In [2]:
print('CONTEXT 1: Parameter Recovery — Ground-truth vs Recovered (N = 28)\n')
published = {'sigma_motor': (0.97, -0.1), 'sigma_spectral': (0.85, 3.0), 'sigma_prior': (0.84, 19.9)}
labels = {'sigma_motor': 'σ_m', 'sigma_spectral': 'σ_mon', 'sigma_prior': 'σ_prior'}

print(f'{"Parameter":<10} {"Pearson r":>10} {"p":>10} {"mean bias":>10}   published')
for param, (r_pub, bias_pub) in published.items():
    true_vals = recovery_df[f'{param}_true'].values
    rec_vals = recovery_df[f'{param}_recovered'].values
    r, p = stats.pearsonr(true_vals, rec_vals)
    bias = np.mean(rec_vals - true_vals)
    print(f'{labels[param]:<10} {r:>10.3f} {p:>10.1e} {bias:>+10.2f}   r = {r_pub}, bias = {bias_pub:+}')
    check('3.3', f'recovery r {labels[param]}', r, r_pub)
    check('3.3', f'recovery mean bias {labels[param]}', bias, bias_pub, decimals=1)

CONTEXT 1: Parameter Recovery — Ground-truth vs Recovered (N = 28)

Parameter   Pearson r          p  mean bias   published
σ_m             0.968    3.9e-17      -0.14   r = 0.97, bias = -0.1
σ_mon           0.853    8.1e-09      +2.98   r = 0.85, bias = +3.0
σ_prior         0.844    1.7e-08     +19.94   r = 0.84, bias = +19.9


In [3]:
print('Inter-parameter correlations (recovered parameters)\n')
for p1, p2, label, r_pub, p_pub in [('sigma_motor', 'sigma_spectral', 'σ_m vs σ_mon', -0.01, 0.95),
                                     ('sigma_spectral', 'sigma_prior', 'σ_mon vs σ_prior', -0.16, 0.42)]:
    r, p = stats.pearsonr(recovery_df[f'{p1}_recovered'], recovery_df[f'{p2}_recovered'])
    print(f'{label:<18} r = {r:+.3f}, p = {p:.3f}   (published r = {r_pub}, p = {p_pub})')
    check('3.3', f'inter-parameter r {label}', r, r_pub)
    check('3.3', f'inter-parameter p {label}', p, p_pub)

# Per-trial NLL SD (Sec. 3.3, computed in 03_nll_stability.ipynb): published "4.31 ± 3.86 to 2.01 ± 0.85,
# mean ± standard error"; the spread is the standard deviation across the 28 configurations.
note('3.3', 'per-trial NLL SD 4.31 ± 3.86 → 2.01 ± 0.85', 'mean ± SE', 'mean ± SD',
     'computed in 03_nll_stability.ipynb; the ± values are SDs across the 28 configurations')

Inter-parameter correlations (recovered parameters)

σ_m vs σ_mon       r = -0.013, p = 0.947   (published r = -0.01, p = 0.95)
σ_mon vs σ_prior   r = -0.159, p = 0.418   (published r = -0.16, p = 0.42)


## Context 2 — Behavioural-Metric Recovery (Sec. 3.3)

Published: LE r = 0.96 (mean difference −0.04° ± 0.79°), PE r = 0.72 (−0.34° ± 6.92°),
QE r = 0.80 (2.08% ± 6.75%).

These values come from the last section of `04_parameter_recovery.ipynb`, which regenerates
responses from the recovered parameters (not cached in `results/`). The values below are copied
from that notebook's saved output.

The published LE r = 0.96 is a stale value: the notebook gives r = 0.97 (0.973 from the 28 printed
value pairs).

In [4]:
# Saved output of 04_parameter_recovery.ipynb ("Localization Metrics: Ground-Truth vs Recovered")
notebook3 = {'LE': (0.97, -0.04, 0.79), 'PE': (0.72, -0.34, 6.92), 'QE': (0.80, 2.08, 6.75)}
published = {'LE': (0.96, -0.04, 0.79), 'PE': (0.72, -0.34, 6.92), 'QE': (0.80, 2.08, 6.75)}
for m in notebook3:
    r, d, sd = notebook3[m]
    print(f'{m}: notebook 04 r = {r:.2f}, diff = {d:+.2f} ± {sd:.2f}   (published r = {published[m][0]:.2f})')
    if m == 'LE':
        note('3.3', 'metric recovery r LE (notebook 04 output)', 0.96, r,
             'stale value in the paper; the analysis gives r = 0.97')
    else:
        check('3.3', f'metric recovery r {m} (notebook 04 output)', r, published[m][0])
    check('3.3', f'metric recovery mean diff {m} (notebook 04 output)', d, published[m][1])

LE: notebook 04 r = 0.97, diff = -0.04 ± 0.79   (published r = 0.96)
PE: notebook 04 r = 0.72, diff = -0.34 ± 6.92   (published r = 0.72)
QE: notebook 04 r = 0.80, diff = +2.08 ± 6.75   (published r = 0.80)


## Context 1b — Template Reconstruction Error (Sec. 3.2)

Source: `results/interpolation_errors.csv` (notebook 01). RMSE between the spectral cues of the
measured HRTF directions and the nearest template direction, averaged over left and right ears,
for 33 participants.

Published: barumerli2023 2.03 ± 0.04 dB, SH 0.49 ± 0.02, SHMAX 0.23 ± 0.01, barycentric 0.24 ± 0.00.
The ± values are standard deviations across participants (the paper says standard error); with
the SD, barycentric reads 0.24 ± 0.01.


In [5]:
print('CONTEXT 1b: Template reconstruction error (RMSE, dB), N = 33\n')
rmse_df = pd.read_csv(paths.RESULTS_DIR / 'interpolation_errors.csv')
published_rmse = {'barumerli2023': (2.03, 0.04), 'SH': (0.49, 0.02), 'SHMAX': (0.23, 0.01), 'barycentric': (0.24, 0.00)}
for method, (m_pub, sd_pub) in published_rmse.items():
    v = rmse_df.loc[rmse_df['method'] == method, 'RMSE_mean']
    print(f'{method:<14} {v.mean():.3f} ± {v.std():.3f} (SD), SE {v.sem():.3f}, range {v.min():.2f}–{v.max():.2f}   '
          f'(published {m_pub} ± {sd_pub})')
    check('3.2', f'RMSE {method} mean', v.mean(), m_pub)
    if method == 'barycentric':
        note('3.2', f'RMSE {method} SD', sd_pub, round(v.std(), 3), 'the SD is 0.008, i.e. 0.01 at two decimals')
    else:
        check('3.2', f'RMSE {method} SD', v.std(), sd_pub)
note('3.2', 'RMSE ± values', 'mean ± SE', 'mean ± SD', 'the ± values in Sec. 3.2 are standard deviations')


CONTEXT 1b: Template reconstruction error (RMSE, dB), N = 33

barumerli2023  2.028 ± 0.038 (SD), SE 0.007, range 1.96–2.13   (published 2.03 ± 0.04)
SH             0.491 ± 0.015 (SD), SE 0.003, range 0.46–0.51   (published 0.49 ± 0.02)
SHMAX          0.233 ± 0.007 (SD), SE 0.001, range 0.22–0.25   (published 0.23 ± 0.01)
barycentric    0.240 ± 0.008 (SD), SE 0.001, range 0.22–0.25   (published 0.24 ± 0.0)


## Context 3 — Fitted Parameters (Sec. 3.4, Table 1 top panel)

Source: `results/fitting_results.csv` (notebook 07); tests as in `R/fitting_analysis.Rmd`.

Published:
- Table 1 (mean ± SE): σ_m 12.5 ± 1.4/1.5; σ_mon 10.4 ± 0.9 (SHMAX), 10.1 ± 0.8 (SH), 9.5 ± 0.6 (barycentric),
  9.5 ± 0.7 (barumerli2023); σ_prior 69.0 ± 9.0, 78.1 ± 9.6, 73.5 ± 10.0, 55.0 ± 6.2
- σ_m range 2.2°–35.7° (median 9.7°)
- σ_mon: Friedman χ²(3) = 5.33, p = .149; Spearman ρ SHMAX–barycentric = 0.81, SHMAX–barumerli2023 = 0.45 (p = .008)
- σ_prior: Friedman χ²(3) = 10.91, p = .012; median within-participant range 22.9°

In [6]:
print('CONTEXT 3: Fitted parameters (N = 33)\n')
table1_top = {  # published mean, SE
    'SHMAX':         {'sigma_motor': (12.5, 1.5), 'sigma_spectral': (10.4, 0.9), 'sigma_prior': (69.0, 9.0)},
    'SH':            {'sigma_motor': (12.5, 1.5), 'sigma_spectral': (10.1, 0.8), 'sigma_prior': (78.1, 9.6)},
    'barycentric':   {'sigma_motor': (12.5, 1.5), 'sigma_spectral': (9.5, 0.6),  'sigma_prior': (73.5, 10.0)},
    'barumerli2023': {'sigma_motor': (12.5, 1.4), 'sigma_spectral': (9.5, 0.7),  'sigma_prior': (55.0, 6.2)},
}
for method, params in table1_top.items():
    sub = fitting_df[fitting_df['method'] == method]
    cells = []
    for param, (m_pub, se_pub) in params.items():
        m, se = sub[param].mean(), sub[param].sem()
        cells.append(f'{param} {m:5.1f} ± {se:4.1f}')
        check('Tab. 1', f'{method} {param} mean', m, m_pub, decimals=1)
        check('Tab. 1', f'{method} {param} SE', se, se_pub, decimals=1)
    print(f'{method:<14}', ' | '.join(cells))

sm = fitting_df['sigma_motor']
print(f'\nσ_m range {sm.min():.1f}–{sm.max():.1f}°, median {sm.median():.1f}°')
check('3.4', 'σ_m min', sm.min(), 2.2, decimals=1)
note('3.4', 'σ_m max', 35.7, round(sm.max(), 2),
     'σ_m is re-estimated for each interpolation method (Monte Carlo noise); the maximum is '
     f'{fitting_df.loc[fitting_df.method == "SHMAX", "sigma_motor"].max():.2f}° for SHMAX and '
     f'{sm.max():.2f}° pooled over methods')
check('3.4', 'σ_m median', sm.median(), 9.7, decimals=1)

for param, chi_pub, p_pub in [('sigma_spectral', 5.33, 0.149), ('sigma_prior', 10.91, 0.012)]:
    wide = fitting_df.pivot(index='participant', columns='method', values=param)[methods]
    chi2, p = stats.friedmanchisquare(*[wide[m] for m in methods])
    print(f'Friedman {param}: χ²(3) = {chi2:.2f}, p = {p:.3f}   (published χ² = {chi_pub}, p = {p_pub})')
    check('3.4', f'Friedman χ² {param}', chi2, chi_pub)
    check('3.4', f'Friedman p {param}', p, p_pub, decimals=3)

prior_range = fitting_df.groupby('participant')['sigma_prior'].agg(lambda x: x.max() - x.min())
print(f'σ_prior median within-participant range: {prior_range.median():.1f}°')
check('3.4', 'σ_prior median within-participant range', prior_range.median(), 22.9, decimals=1)

CONTEXT 3: Fitted parameters (N = 33)

SHMAX          sigma_motor  12.5 ±  1.5 | sigma_spectral  10.4 ±  0.9 | sigma_prior  69.0 ±  9.0
SH             sigma_motor  12.5 ±  1.5 | sigma_spectral  10.1 ±  0.7 | sigma_prior  78.1 ±  9.6
barycentric    sigma_motor  12.5 ±  1.5 | sigma_spectral   9.5 ±  0.6 | sigma_prior  73.5 ± 10.0
barumerli2023  sigma_motor  12.5 ±  1.4 | sigma_spectral   9.5 ±  0.7 | sigma_prior  55.0 ±  6.2

σ_m range 2.2–35.8°, median 9.7°
Friedman sigma_spectral: χ²(3) = 5.33, p = 0.149   (published χ² = 5.33, p = 0.149)
Friedman sigma_prior: χ²(3) = 10.91, p = 0.012   (published χ² = 10.91, p = 0.012)
σ_prior median within-participant range: 22.9°


True

In [7]:
print('Between-method Spearman correlations (N = 33)\n')
for param in ['sigma_spectral', 'sigma_prior']:
    wide = fitting_df.pivot(index='participant', columns='method', values=param)
    print(f'--- {param} ---')
    for i in range(len(methods)):
        for j in range(i + 1, len(methods)):
            rho, p = stats.spearmanr(wide[methods[i]], wide[methods[j]])
            print(f'  {methods[i]}–{methods[j]:<15} ρ = {rho:.3f}, p = {p:.4f}')

spec_wide = fitting_df.pivot(index='participant', columns='method', values='sigma_spectral')
for m1, m2, rho_pub, p_pub in [('SHMAX', 'barycentric', 0.81, None), ('SHMAX', 'barumerli2023', 0.45, 0.008)]:
    rho, p = stats.spearmanr(spec_wide[m1], spec_wide[m2])
    check('3.4', f'σ_mon Spearman ρ {m1}–{m2}', rho, rho_pub)
    if p_pub is not None:
        check('3.4', f'σ_mon Spearman p {m1}–{m2}', p, p_pub, 3)

Between-method Spearman correlations (N = 33)

--- sigma_spectral ---
  SHMAX–SH              ρ = 0.711, p = 0.0000
  SHMAX–barycentric     ρ = 0.809, p = 0.0000
  SHMAX–barumerli2023   ρ = 0.453, p = 0.0081
  SH–barycentric     ρ = 0.728, p = 0.0000
  SH–barumerli2023   ρ = 0.569, p = 0.0005
  barycentric–barumerli2023   ρ = 0.484, p = 0.0043
--- sigma_prior ---
  SHMAX–SH              ρ = 0.790, p = 0.0000
  SHMAX–barycentric     ρ = 0.897, p = 0.0000
  SHMAX–barumerli2023   ρ = 0.821, p = 0.0000
  SH–barycentric     ρ = 0.726, p = 0.0000
  SH–barumerli2023   ρ = 0.733, p = 0.0000
  barycentric–barumerli2023   ρ = 0.828, p = 0.0000


## Context 4 — Posterior Predictive Checks (Sec. 3.4.1, Table 1 middle/bottom, Fig. 5)

Source: `results/synthetic_vs_actual_comparison.csv` (notebook 08).

Published:
- Table 1 middle panel (mean ± SE): Actual LE 12.7 ± 0.9, PE 33.8 ± 1.1, QE 15.1 ± 1.8; model values per method
- Table 1 bottom panel (Pearson r): LE 0.89 (all methods); PE 0.63 / 0.57 / 0.65 / 0.65; QE 0.52 / 0.51 / 0.60 / 0.19
  (SHMAX / SH / barycentric / barumerli2023); barumerli2023 QE p = .30
- no group-level bias: Wilcoxon p > .27 for LE, PE and QE (this holds for SHMAX; over all methods the
  smallest p is .13, still not significant)
- lateral accuracy bias ≈ 3° (Wilcoxon p = .002); polar accuracy: no difference (p > .26), r < 0.22 (p > .21)

In [8]:
print('CONTEXT 4: Posterior Predictive Checks — Actual vs Synthetic\n')
actual = comparison_df[comparison_df['data_type'] == 'actual'].set_index('participant')
synth = comparison_df[comparison_df['data_type'] == 'synthetic']

def paired(method, col):
    s = synth[synth['method'] == method].set_index('participant')[col]
    a = actual.loc[s.index, col]
    return a.values, s.values

metrics_check = [('rmsL_deg', 'LE'), ('rmsPmedianlocal_deg', 'PE'), ('querrMiddlebrooks', 'QE')]
table1_mid = {'Actual': (12.7, 0.9, 33.8, 1.1, 15.1, 1.8),
              'SHMAX': (13.3, 1.2, 33.7, 0.8, 16.4, 1.3), 'SH': (13.5, 1.2, 33.9, 0.8, 16.4, 1.1),
              'barycentric': (13.3, 1.2, 33.4, 0.8, 15.5, 1.1), 'barumerli2023': (13.6, 1.2, 34.3, 0.7, 17.4, 1.1)}
table1_r = {'SHMAX': (0.89, 0.63, 0.52), 'SH': (0.89, 0.57, 0.51),
            'barycentric': (0.89, 0.65, 0.60), 'barumerli2023': (0.89, 0.65, 0.19)}

for k, (col, label) in enumerate(metrics_check):
    vals = actual[col]
    m_pub, se_pub = table1_mid['Actual'][2 * k], table1_mid['Actual'][2 * k + 1]
    check('Tab. 1', f'Actual {label} mean', vals.mean(), m_pub, 1)
    check('Tab. 1', f'Actual {label} SE', vals.sem(), se_pub, 1)

wilcox_p = []
print(f'{"Method":<14} {"Metric":<4} {"mean ± SE":>13} {"r":>6} {"p(r)":>8} {"Wilcoxon p":>11}')
for method in methods:
    for k, (col, label) in enumerate(metrics_check):
        a, s = paired(method, col)
        r, p = stats.pearsonr(a, s)
        w_p = stats.wilcoxon(a, s).pvalue
        wilcox_p.append(w_p)
        se = np.std(s) / np.sqrt(len(s))   # population SD / sqrt(N), as in notebook 08
        print(f'{method:<14} {label:<4} {s.mean():6.1f} ± {se:4.1f} {r:6.2f} {p:8.4f} {w_p:11.3f}')
        check('Tab. 1', f'{method} {label} mean', s.mean(), table1_mid[method][2 * k], 1)
        check('Tab. 1', f'{method} {label} SE', se, table1_mid[method][2 * k + 1], 1)
        check('Tab. 1', f'{method} {label} r', r, table1_r[method][k])

a, s = paired('barumerli2023', 'querrMiddlebrooks')
p_qe = stats.pearsonr(a, s)[1]
check('3.4.1', 'barumerli2023 QE p(r)', p_qe, 0.30)
# Published: "no significant bias (Wilcoxon p > .27 for LE, PE, and QE)"
p_shmax = wilcox_p[:3]   # SHMAX is the first method
print(f'\nWilcoxon p, SHMAX (LE, PE, QE): {", ".join(f"{p:.3f}" for p in p_shmax)}; '
      f'min over all methods: {min(wilcox_p):.3f}  (published p > .27)')
summary_rows.append({'Section': '3.4.1', 'Quantity': 'min Wilcoxon p (LE, PE, QE), SHMAX', 'Published': '> .27',
                     'Computed': round(min(p_shmax), 3), 'Match': '✓' if round(min(p_shmax), 2) >= 0.27 else '✗'})
note('3.4.1', 'min Wilcoxon p (LE, PE, QE), all methods', '> .27', round(min(wilcox_p), 3),
     'p > .27 holds for SHMAX only; over all methods the smallest p is .13 (no significant bias)')

print('\nAccuracy metrics (±30° cutoff), model − actual')
acc = {}
for col, label in [('accL_cutoff_30_deg', 'lateral bias'), ('accP_cutoff_30_deg', 'polar bias')]:
    for method in methods:
        a, s = paired(method, col)
        r, p = stats.pearsonr(a, s)
        w_p = stats.wilcoxon(a, s).pvalue
        acc.setdefault(label, []).append((w_p, r, p))
        print(f'  {label:<13} {method:<14} actual {a.mean():+5.2f}°, model {s.mean():+5.2f}°, '
              f'Wilcoxon p = {w_p:.4f}, r = {r:+.2f} (p = {p:.2f})')
lat_p = acc['lateral bias'][methods.index('SHMAX')][0]
check('3.4.1', 'lateral bias Wilcoxon p (SHMAX)', lat_p, 0.002, 3)
pol = acc['polar bias']
for quantity, value, ok in [('polar bias Wilcoxon p > .26 (min over methods)', min(w for w, _, _ in pol), min(w for w, _, _ in pol) >= 0.255),
                            ('polar bias r < 0.22 (max over methods)', max(r for _, r, _ in pol), round(max(r for _, r, _ in pol), 2) <= 0.22),
                            ('polar bias p(r) > .21 (min over methods)', min(p for _, _, p in pol), round(min(p for _, _, p in pol), 2) >= 0.21)]:
    summary_rows.append({'Section': '3.4.1', 'Quantity': quantity, 'Published': quantity.split(' (')[0].split()[-2] + ' ' + quantity.split(' (')[0].split()[-1],
                         'Computed': round(value, 3), 'Match': '✓' if ok else '✗'})

CONTEXT 4: Posterior Predictive Checks — Actual vs Synthetic

Method         Metric     mean ± SE      r     p(r)  Wilcoxon p
SHMAX          LE     13.3 ±  1.2   0.89   0.0000       0.416
SHMAX          PE     33.7 ±  0.8   0.63   0.0001       0.646
SHMAX          QE     16.4 ±  1.3   0.52   0.0017       0.272
SH             LE     13.5 ±  1.2   0.89   0.0000       0.264
SH             PE     33.9 ±  0.8   0.57   0.0006       0.944
SH             QE     16.4 ±  1.1   0.51   0.0022       0.358
barycentric    LE     13.3 ±  1.2   0.89   0.0000       0.416
barycentric    PE     33.4 ±  0.8   0.65   0.0000       0.447
barycentric    QE     15.5 ±  1.1   0.60   0.0002       0.513
barumerli2023  LE     13.6 ±  1.2   0.89   0.0000       0.130
barumerli2023  PE     34.3 ±  0.7   0.65   0.0000       0.738
barumerli2023  QE     17.4 ±  1.1   0.19   0.2988       0.183

Wilcoxon p, SHMAX (LE, PE, QE): 0.416, 0.646, 0.272; min over all methods: 0.130  (published p > .27)

Accuracy metrics (±30° cut

In [9]:
# barumerli2023 QE: r = 0.19, p = .30 (N = 33) — the template artefacts prevent tracking individual QE
a, s = paired('barumerli2023', 'querrMiddlebrooks')
r, p = stats.pearsonr(a, s)
print(f'barumerli2023 QE: Pearson r = {r:.3f}, p = {p:.4f}, N = {len(a)}  (published r = 0.19, p = .30)')
for method in ['SHMAX', 'SH', 'barycentric']:
    a, s = paired(method, 'querrMiddlebrooks')
    r, p = stats.pearsonr(a, s)
    print(f'{method:<13} QE: Pearson r = {r:.3f}, p = {p:.4f}')

barumerli2023 QE: Pearson r = 0.186, p = 0.2988, N = 33  (published r = 0.19, p = .30)
SHMAX         QE: Pearson r = 0.524, p = 0.0017
SH            QE: Pearson r = 0.515, p = 0.0022
barycentric   QE: Pearson r = 0.601, p = 0.0002


## Context 5 — Model Comparison (Sec. 3.5, Table 2, Fig. 6)

BIC = 3 log(N) + 2 NLL (500 Monte Carlo samples; `nll_500`, N = `n_trials`).

Published:
- mean ΔBIC to the best method per participant: SHMAX 25.8, barycentric 32.6, SH 42.4, barumerli2023 45.2 (Fig. 6)
- Friedman test on BIC across methods: χ²(3) = 5.04, p = .169
- Table 2 (vs barumerli2023): ΔBIC SHMAX −19.4 ± 10.1, barycentric −12.5 ± 16.7, SH −2.8 ± 15.3;
  wins 19/33, 18/33, 15/33; very strong (ΔBIC < −10) 17, 15, 13. "Wins" counts participants with at least
  positive evidence for the alternative method (ΔBIC < −2), not merely a lower BIC as the caption says
- SHMAX vs barumerli2023: 17 very strong, 2 positive, 4 negligible, 10 favouring barumerli2023

In [10]:
print('CONTEXT 5: Model comparison\n')
df = fitting_df.copy()
df['BIC'] = 3 * np.log(df['n_trials']) + 2 * df['nll_500']
df['delta_best'] = df['BIC'] - df.groupby('participant')['BIC'].transform('min')

for method, pub in [('SHMAX', 25.8), ('barycentric', 32.6), ('SH', 42.4), ('barumerli2023', 45.2)]:
    m = df.loc[df['method'] == method, 'delta_best'].mean()
    print(f'mean ΔBIC to best, {method:<14}: {m:5.1f}  (published {pub})')
    check('3.5', f'mean ΔBIC to best {method}', m, pub, 1)

bic_wide = df.pivot(index='participant', columns='method', values='BIC')[methods]
chi2, p = stats.friedmanchisquare(*[bic_wide[m] for m in methods])
print(f'\nFriedman on BIC: χ²(3) = {chi2:.2f}, p = {p:.3f}  (published χ² = 5.04, p = .169)')
check('3.5', 'Friedman χ² BIC', chi2, 5.04)
check('3.5', 'Friedman p BIC', p, 0.169, 3)

print('\nTable 2 (relative to barumerli2023; negative favours the alternative method)')
table2 = {'SHMAX': (-19.4, 10.1, 19, 17), 'barycentric': (-12.5, 16.7, 18, 15), 'SH': (-2.8, 15.3, 15, 13)}
for method, (m_pub, se_pub, wins_pub, vs_pub) in table2.items():
    d = (bic_wide[method] - bic_wide['barumerli2023']).values
    m, se = d.mean(), d.std() / np.sqrt(len(d))
    # "Wins" as computed for Table 2 in notebook 08: at least positive evidence (ΔBIC ≤ −2);
    # a strict count of lower BIC (ΔBIC < 0) is printed for reference
    wins, very_strong = int(np.sum(d <= -2)), int(np.sum(d < -10))
    print(f'{method:<12} ΔBIC {m:6.1f} ± {se:4.1f}   wins (ΔBIC ≤ −2) {wins}/33 '
          f'[ΔBIC < 0: {int(np.sum(d < 0))}/33]   very strong {very_strong}')
    check('Tab. 2', f'{method} ΔBIC mean', m, m_pub, 1)
    check('Tab. 2', f'{method} ΔBIC SE', se, se_pub, 1)
    check('Tab. 2', f'{method} wins (ΔBIC ≤ −2)', wins, wins_pub, 0)
    note('Tab. 2', f'{method} wins: caption definition (lower BIC)', wins_pub, int(np.sum(d < 0)),
         'the caption says "lower BIC"; the published counts use ΔBIC < −2')
    check('Tab. 2', f'{method} very strong', very_strong, vs_pub, 0)

d = (bic_wide['barumerli2023'] - bic_wide['SHMAX']).values   # positive = SHMAX better
cats = {'very strong SHMAX (>10)': (int(np.sum(d > 10)), 17),
        'positive SHMAX (2–10)': (int(np.sum((d > 2) & (d <= 10))), 2),
        'negligible (|Δ| ≤ 2)': (int(np.sum(np.abs(d) <= 2)), 4),
        'favouring barumerli2023 (< −2)': (int(np.sum(d < -2)), 10)}
print('\nSHMAX vs barumerli2023 (Kass & Raftery categories)')
for k, (n, pub) in cats.items():
    print(f'  {k:<32} {n:3d}  (published {pub})')
    check('3.5', f'SHMAX vs barumerli2023: {k}', n, pub, 0)

CONTEXT 5: Model comparison

mean ΔBIC to best, SHMAX         :  25.8  (published 25.8)
mean ΔBIC to best, barycentric   :  32.6  (published 32.6)
mean ΔBIC to best, SH            :  42.4  (published 42.4)
mean ΔBIC to best, barumerli2023 :  45.2  (published 45.2)

Friedman on BIC: χ²(3) = 5.04, p = 0.169  (published χ² = 5.04, p = .169)

Table 2 (relative to barumerli2023; negative favours the alternative method)
SHMAX        ΔBIC  -19.4 ± 10.1   wins (ΔBIC ≤ −2) 19/33 [ΔBIC < 0: 21/33]   very strong 17
barycentric  ΔBIC  -12.5 ± 16.7   wins (ΔBIC ≤ −2) 18/33 [ΔBIC < 0: 19/33]   very strong 15
SH           ΔBIC   -2.8 ± 15.3   wins (ΔBIC ≤ −2) 15/33 [ΔBIC < 0: 16/33]   very strong 13

SHMAX vs barumerli2023 (Kass & Raftery categories)
  very strong SHMAX (>10)           17  (published 17)
  positive SHMAX (2–10)              2  (published 2)
  negligible (|Δ| ≤ 2)               4  (published 4)
  favouring barumerli2023 (< −2)    10  (published 10)


## Summary

All published values checked above, with the value recomputed from `results/`. A ✗ marks a value
that does not match the published one at the published precision; **note** marks a documented
difference (see the README, "Notes on the published values").

In [11]:
summary_df = pd.DataFrame(summary_rows)
if 'Note' not in summary_df:
    summary_df['Note'] = ''
summary_df['Note'] = summary_df['Note'].fillna('')
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', 120)
print(summary_df.drop(columns='Note').to_string(index=False))
n_ok = (summary_df['Match'] == '✓').sum()
n_bad = (summary_df['Match'] == '✗').sum()
n_note = (summary_df['Match'] == 'note').sum()
print(f'\n{n_ok} match, {n_bad} do not match, {n_note} documented notes ({len(summary_df)} values)')
if n_note:
    print('\nDocumented notes:')
    for _, row in summary_df[summary_df['Match'] == 'note'].iterrows():
        print(f"  [{row['Section']}] {row['Quantity']}: published {row['Published']}, computed {row['Computed']} -- {row['Note']}")
if n_bad:
    print('\nMismatches:')
    print(summary_df[summary_df['Match'] == '✗'].drop(columns='Note').to_string(index=False))


Section                                               Quantity Published  Computed Match
    3.3                                         recovery r σ_m      0.97     0.968     ✓
    3.3                                 recovery mean bias σ_m      -0.1     -0.14     ✓
    3.3                                       recovery r σ_mon      0.85     0.853     ✓
    3.3                               recovery mean bias σ_mon       3.0      2.98     ✓
    3.3                                     recovery r σ_prior      0.84     0.844     ✓
    3.3                             recovery mean bias σ_prior      19.9     19.94     ✓
    3.3                         inter-parameter r σ_m vs σ_mon     -0.01    -0.013     ✓
    3.3                         inter-parameter p σ_m vs σ_mon      0.95     0.947     ✓
    3.3                     inter-parameter r σ_mon vs σ_prior     -0.16    -0.159     ✓
    3.3                     inter-parameter p σ_mon vs σ_prior      0.42     0.418     ✓
    3.3             p